# 01 · EDA de la OBT ECU 911

Predicción de demanda alta a t+3. Se inspecciona el snapshot de Snowflake y se explora solo train para decidir sobre features. La ausencia de datos no se reemplaza por simulaciones.

**Orden:** extraer snapshot una vez con `python -m src.datos snapshot`; luego ejecutar estos cinco notebooks con kernels limpios.

In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.datos import DATA, DOCS, EVIDENCE
from src.datos import execute
HAS_DATA = (DATA / "obt.parquet").exists()
display(Markdown("**OBT real local disponible**" if HAS_DATA else
    "**PENDIENTE:** faltan credenciales Snowflake válidas y snapshot. Este notebook no contiene resultados empíricos del ECU 911."))


## Integridad y disponibilidad

`load_snapshot` verifica SHA256, grain cantón-día, spine diario, medias/lag y targets futuros. El primer año calibra P90; el 20% final se reserva sin inspeccionar distribuciones del hold-out. No se imprimen credenciales.

In [ ]:
if HAS_DATA:
    from src.datos import load_snapshot
    obt, snapshot = load_snapshot()
    display(pd.DataFrame([snapshot]))
    print("Tamaño OBT:", obt.shape)
    from src.datos import context
    ctx = context()
    display(pd.DataFrame([ctx["split"]]))
    display(pd.DataFrame([{ "fold": k, "train_rows": len(a), "valid_rows": len(b),
        "train_end": str(ctx["train"].iloc[a].fecha.max()),
        "valid_start": str(ctx["train"].iloc[b].fecha.min()) }
        for k, (a, b) in enumerate(ctx["folds"])]))

## Tipos, faltantes, descriptivos, target y outliers

Se registra IQR y |Z|≥3; no se eliminan automáticamente los picos que queremos anticipar. Spearman mide asociación monótona con conteos asimétricos. El target binario también se analiza mediante prevalencia para t+3.

In [ ]:
if HAS_DATA:
    eda = execute("eda")
    print("Perfil estructural de la OBT completa (incluye bordes con ventanas/targets incompletos):")
    display(pd.read_csv(EVIDENCE / "eda/snapshot_schema.csv"))
    print("Perfil del bloque de entrenamiento utilizado en EDA y modelado:")
    display(pd.read_csv(EVIDENCE / "eda/schema.csv"))
    display(pd.read_csv(EVIDENCE / "eda/descriptive.csv"))
    display(pd.read_csv(EVIDENCE / "eda/outliers.csv"))
    display(pd.Series(eda["prevalence"], name="fracción alta demanda"))
    display(Image(filename=str(EVIDENCE / "eda/distribucion_boxplot.png")))
    display(Image(filename=str(EVIDENCE / "eda/correlacion.png")))

## Hallazgos que cambian el modelado

La tabla se genera con evidencia real de train; hasta entonces no se afirma que se hayan observado tres hallazgos. Anomalía enero 2024 documentada por PSet 2: se conserva y su marca no se ofrece al modelo.

In [ ]:
if HAS_DATA:
    display(pd.read_csv(EVIDENCE / "eda/hallazgos.csv"))
else:
    display(Markdown("Pendiente: completar EDA con la OBT real y revisar sus consecuencias antes de entrenar."))